# MP-DATA-007: Market Identity Review & Panel Freeze Evaluation

**Kernel ID:** `theankitchoubey/mp-data-007-identity-review`  
**Task ID:** `MP-DATA-007` (Step 2: Market Identity Policy & Dataset-Freeze Readiness)  
**Execution Environment:** Kaggle CPU, Internet Off, Private Execution  
**Dataset Input:** `theankitchoubey/mandi-prediction-audit-003b` (v1)

---

### Objectives & Hard-Stop Invariants:
1. **Legacy Panel v2 Reproduction & Integrity Gate:**
   - Ingest raw Agmarknet arrivals using legacy alias table `data/reference/market_aliases.csv`.
   - Assert SHA-256 == `851e718468d6b8c9fd07944535a462338e71aa105af25339fe9438b697088f31`.
   - If hash differs, **HALT EXECUTION IMMEDIATELY**.
2. **Candidate Panel v2.1 Side-by-Side Evaluation:**
   - Ingest using candidate alias table `data/reference/market_alias_candidates_v21.csv`.
   - Compute independent candidate SHA-256 (never forced).
   - Side-by-side comparison: dispositions, district totals, unique natural keys, unique series, rescued/deduplicated/conflicting rows, Excel match classifications, and 77 disputed keys.
3. **Identity Adjudication:**
   - Purwa (Unnao): 205-day hiatus, 0 overlapping days, kept separate (`Purwa` vs `Purwa APMC`).
   - Jhansi Grain (Jhansi): Transitive concordance via `Jhansi (Grain) APMC` (23 keys, 100% agreement) mapped to `Jhansi (Grain)`.
   - Baripaal (Kanpur): 52-day co-reporting, 20 conflicting keys (4.33% divergence), kept separate (`Baripaal APMC` vs `Varipaal`).
4. **Causal Sensitivity & Train-Only Eligibility:**
   - Test interaction of `<20` full-history floor with `>=60` train-only eligibility reconstructed BEFORE the floor.
   - Test causal as-of information availability (withholding post-cutoff evidence in verified vs assumed modes).
   - Dynamically reclassify discovered in-window Excel rows across panel v2 and panel v2.1.
5. **No ML Training:** Purely diagnostic and data policy verification.


In [ ]:
# Cell 1: Environment, Library Versions, Bootstrap Modules, and Input Discovery
import os
import sys
import glob
import json
import base64
import hashlib
import time
import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import scipy
import sklearn
import openpyxl

print("=== ENVIRONMENT & LIBRARY VERSIONS ===")
print(f"Python: {sys.version}")
print(f"Platform: {sys.platform}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"scipy: {scipy.__version__}")
print(f"sklearn: {sklearn.__version__}")
print(f"openpyxl: {openpyxl.__version__}")

np.random.seed(42)
print("Random seed initialized to 42.")

working_root = Path("/kaggle/working") if os.path.exists("/kaggle") else Path(os.environ.get("MANDI_WORKING_ROOT", "."))
src_dir = working_root / "src"
ref_dir = working_root / "data" / "reference"
out_dir = working_root / "experiments" / "outputs" / "007"
scorecards_dir = working_root / "experiments" / "reports" / "scorecards"

src_dir.mkdir(parents=True, exist_ok=True)
ref_dir.mkdir(parents=True, exist_ok=True)
out_dir.mkdir(parents=True, exist_ok=True)
scorecards_dir.mkdir(parents=True, exist_ok=True)

if not (src_dir / "__init__.py").exists():
    (src_dir / "__init__.py").write_text("")

b64_payloads = {
    src_dir / "market_identity.py": "...",
    src_dir / "identity_comparison.py": "...",
    src_dir / "data_pipeline.py": "...",
    src_dir / "ingest_agmarknet_arrivals.py": "...",
    src_dir / "baselines.py": "...",
    ref_dir / "market_aliases.csv": "ZGlzdHJpY3QsbWFya2V0X3JhdyxtYXJrZXRfY2Fub25pY2FsLHJ1bGUsZXZpZGVuY2UsZmlyc3RfZGF0ZSxsYXN0X2RhdGUsb3ZlcmxhcF9kYXlzLHByaWNlX2FncmVlbWVudCxzdGF0dXMKR2hhemlhYmFkLEdoYXppYWJhZCxHaGF6aWFiYWQsSWRlbnRpdHkgKGNhbm9uaWNhbCBhbmNob3IpLENhbm9uaWNhbCBiYXNlbGluZSBhbmNob3IgZm9yIG5vcm1hbGl6ZWQgdG9rZW4sMjAyMS0wOS0wMSwyMDI1LTExLTA0LDAsLEFDQ0VQVEVECkdoYXppYWJhZCxHaGF6aWFiYWQgQVBNQyxHaGF6aWFiYWQsIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTQgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjUtMTEtMDEsMjAyNi0wOS0zMCw0LCxBQ0NFUFRFRApHaGF6aWFiYWQsSGFwdXIsSGFwdXIsSWRlbnRpdHkgKGNhbm9uaWNhbCBhbmNob3IpLENhbm9uaWNhbCBiYXNlbGluZSBhbmNob3IgZm9yIG5vcm1hbGl6ZWQgdG9rZW4sMjAyMS0wOS0wMSwyMDI1LTExLTA1LDAsLEFDQ0VQVEVECkdoYXppYWJhZCxIYXB1ciBBUE1DLEhhcHVyLCJSdWxlIEEgKGNocm9ub2xvZ2ljYWwgdHJhbnNpdGlvbiwgb3ZlcmxhcCA8PSA1ZCkiLG92ZXJsYXBfZGF5cz01IDw9IDUgKGNocm9ub2xvZ2ljYWwgbWlncmF0aW9uKSwyMDI1LTExLTAxLDIwMjYtMDktMzAsNSwsQUNDRVBURUQKR2hhemlhYmFkLE11cmFkbmFnYXIsTXVyYWRuYWdhcixJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIGJhc2VsaW5lIGFuY2hvciBmb3Igbm9ybWFsaXplZCB0b2tlbiwyMDIxLTA5LTAxLDIwMjUtMTEtMDQsMCwsQUNDRVBURUQKR2hhemlhYmFkLE11cmFkbmFnYXIgQVBNQyxNdXJhZG5hZ2FyLCJSdWxlIEEgKGNocm9ub2xvZ2ljYWwgdHJhbnNpdGlvbiwgb3ZlcmxhcCA8PSA1ZCkiLG92ZXJsYXBfZGF5cz00IDw9IDUgKGNocm9ub2xvZ2ljYWwgbWlncmF0aW9uKSwyMDI1LTExLTAxLDIwMjYtMDktMzAsNCwsQUNDRVBURUQKR2hhemlhYmFkLE5vaWRhLE5vaWRhLElkZW50aXR5IChjYW5vbmljYWwgYW5jaG9yKSxDYW5vbmljYWwgYmFzZWxpbmUgYW5jaG9yIGZvciBub3JtYWxpemVkIHRva2VuLDIwMjEtMDktMDIsMjAyNS0xMS0wMywwLCxBQ0NFUFRFRApHaGF6aWFiYWQsTm9pZGEgQVBNQyxOb2lkYSwiUnVsZSBBIChjaHJvbm9sb2dpY2FsIHRyYW5zaXRpb24sIG92ZXJsYXAgPD0gNWQpIixvdmVybGFwX2RheXM9MyA8PSA1IChjaHJvbm9sb2dpY2FsIG1pZ3JhdGlvbiksMjAyNS0xMS0wMSwyMDI2LTA5LTMwLDMsLEFDQ0VQVEVECkpoYW5zaSxCYXJ1d2FzYWdhcixCYXJ1d2FzYWdhcixJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIGJhc2VsaW5lIGFuY2hvciBmb3Igbm9ybWFsaXplZCB0b2tlbiwyMDIxLTA5LTAyLDIwMjUtMTEtMDUsMCwsQUNDRVBURUQKSmhhbnNpLEJhcnV3YXNhZ2FyIEFQTUMsQmFydXdhc2FnYXIsIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTMgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjUtMTEtMDMsMjAyNi0wOS0zMCwzLCxBQ0NFUFRFRApKaGFuc2ksQ2hpcmdhb24sQ2hpcmdhb24sSWRlbnRpdHkgKGNhbm9uaWNhbCBhbmNob3IpLENhbm9uaWNhbCBiYXNlbGluZSBhbmNob3IgZm9yIG5vcm1hbGl6ZWQgdG9rZW4sMjAyMS0wOS0wMSwyMDI1LTExLTA0LDAsLEFDQ0VQVEVECkpoYW5zaSxDaGlyZ2FvbiBBUE1DLENoaXJnYW9uLCJSdWxlIEEgKGNocm9ub2xvZ2ljYWwgdHJhbnNpdGlvbiwgb3ZlcmxhcCA8PSA1ZCkiLG92ZXJsYXBfZGF5cz0zIDw9IDUgKGNocm9ub2xvZ2ljYWwgbWlncmF0aW9uKSwyMDI1LTExLTAyLDIwMjYtMDktMzAsMywsQUNDRVBURUQKSmhhbnNpLEd1cnVzYXJhaSxHdXJ1c2FyYWksSWRlbnRpdHkgKGNhbm9uaWNhbCBhbmNob3IpLENhbm9uaWNhbCBiYXNlbGluZSBhbmNob3IgZm9yIG5vcm1hbGl6ZWQgdG9rZW4sMjAyMS0wOS0wMSwyMDI1LTExLTA0LDAsLEFDQ0VQVEVECkpoYW5zaSxHdXJ1c2FyYWkgQVBNQyxHdXJ1c2FyYWksIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTQgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjUtMTEtMDEsMjAyNi0wOS0zMCw0LCxBQ0NFUFRFRApKaGFuc2ksSmhhbnNpLEpoYW5zaSxJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIGJhc2VsaW5lIGFuY2hvciBmb3Igbm9ybWFsaXplZCB0b2tlbiwyMDIxLTA5LTAxLDIwMjUtMTEtMDQsMCwsQUNDRVBURUQKSmhhbnNpLEpoYW5zaSBBUE1DLEpoYW5zaSwiUnVsZSBBIChjaHJvbm9sb2dpY2FsIHRyYW5zaXRpb24sIG92ZXJsYXAgPD0gNWQpIixvdmVybGFwX2RheXM9NCA8PSA1IChjaHJvbm9sb2dpY2FsIG1pZ3JhdGlvbiksMjAyNS0xMS0wMSwyMDI2LTA5LTMwLDQsLEFDQ0VQVEVECkpoYW5zaSxKaGFuc2kgKEdyYWluKSxKaGFuc2kgKEdyYWluKSxJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIGJhc2VsaW5lIGFuY2hvciBmb3Igbm9ybWFsaXplZCB0b2tlbiwyMDI0LTA1LTI1LDIwMjUtMTEtMDMsMCwsQUNDRVBURUQKSmhhbnNpLEpoYW5zaSAoR3JhaW4pIEFQTUMsSmhhbnNpIChHcmFpbiksIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTIgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjUtMTEtMDIsMjAyNi0wMS0xNCwyLCxBQ0NFUFRFRApKaGFuc2ksSmhhbnNpIEdyYWluIEFQTUMsSmhhbnNpIChHcmFpbiksIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTAgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjYtMDEtMDEsMjAyNi0wMS0xNCwwLCxBQ0NFUFRFRApKaGFuc2ksTWF1cmFuaXB1cixNYXVyYW5pcHVyLElkZW50aXR5IChjYW5vbmljYWwgYW5jaG9yKSxDYW5vbmljYWwgYmFzZWxpbmUgYW5jaG9yIGZvciBub3JtYWxpemVkIHRva2VuLDIwMjEtMDktMDEsMjAyNS0xMS0wNCwwLCxBQ0NFUFRFRApKaGFuc2ksTWF1cmFuaXB1ciBBUE1DLE1hdXJhbmlwdXIsIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTQgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjUtMTEtMDEsMjAyNi0wOS0zMCw0LCxBQ0NFUFRFRApKaGFuc2ksTW90aCxNb3RoLElkZW50aXR5IChjYW5vbmljYWwgYW5jaG9yKSxDYW5vbmljYWwgYmFzZWxpbmUgYW5jaG9yIGZvciBub3JtYWxpemVkIHRva2VuLDIwMjEtMDktMDEsMjAyNS0xMS0wNCwwLCxBQ0NFUFRFRApKaGFuc2ksTW90aCBBUE1DLE1vdGgsIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTQgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjUtMTEtMDEsMjAyNi0wOS0zMCw0LCxBQ0NFUFRFRApLYW5wdXIsQmFyaXBhYWwgQVBNQyxCYXJpcGFhbCBBUE1DLElkZW50aXR5IChzaW5nbGUgdG9rZW4pLFVuaXF1ZSBtYXJrZXQgZm9yIG5vcm1hbGl6ZWQgdG9rZW4gaW4gZGlzdHJpY3QsMjAyNi0wMy0wMSwyMDI2LTA5LTMwLDAsLEFDQ0VQVEVECkthbnB1cixDaG91YmVwdXIsQ2hvdWJlcHVyLElkZW50aXR5IChjYW5vbmljYWwgYW5jaG9yKSxDYW5vbmljYWwgYmFzZWxpbmUgYW5jaG9yIGZvciBub3JtYWxpemVkIHRva2VuLDIwMjEtMDktMDEsMjAyNS0xMS0wNCwwLCxBQ0NFUFRFRApLYW5wdXIsQ2hvdWJlcHVyIEFQTUMsQ2hvdWJlcHVyLCJSdWxlIEEgKGNocm9ub2xvZ2ljYWwgdHJhbnNpdGlvbiwgb3ZlcmxhcCA8PSA1ZCkiLG92ZXJsYXBfZGF5cz00IDw9IDUgKGNocm9ub2xvZ2ljYWwgbWlncmF0aW9uKSwyMDI1LTExLTAxLDIwMjYtMDktMzAsNCwsQUNDRVBURUQKS2FucHVyLEthbnB1cihHcmFpbiksS2FucHVyKEdyYWluKSxJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIGJhc2VsaW5lIGFuY2hvciBmb3Igbm9ybWFsaXplZCB0b2tlbiwyMDIxLTA5LTAxLDIwMjUtMTEtMDQsMCwsQUNDRVBURUQKS2FucHVyLEthbnB1cihHcmFpbikgQVBNQyxLYW5wdXIoR3JhaW4pLCJSdWxlIEEgKGNocm9ub2xvZ2ljYWwgdHJhbnNpdGlvbiwgb3ZlcmxhcCA8PSA1ZCkiLG92ZXJsYXBfZGF5cz00IDw9IDUgKGNocm9ub2xvZ2ljYWwgbWlncmF0aW9uKSwyMDI1LTExLTAxLDIwMjYtMDktMzAsNCwsQUNDRVBURUQKS2FucHVyLFV0dGFyaXB1cmEsVXR0YXJpcHVyYSxJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIGJhc2VsaW5lIGFuY2hvciBmb3Igbm9ybWFsaXplZCB0b2tlbiwyMDIxLTEwLTI0LDIwMjUtMTEtMDQsMCwsQUNDRVBURUQKS2FucHVyLFV0dGFyaXB1cmEgQVBNQyxVdHRhcmlwdXJhLCJSdWxlIEEgKGNocm9ub2xvZ2ljYWwgdHJhbnNpdGlvbiwgb3ZlcmxhcCA8PSA1ZCkiLG92ZXJsYXBfZGF5cz00IDw9IDUgKGNocm9ub2xvZ2ljYWwgbWlncmF0aW9uKSwyMDI1LTExLTAxLDIwMjYtMDktMzAsNCwsQUNDRVBURUQKS2FucHVyLFZhcmlwYWFsLFZhcmlwYWFsLElkZW50aXR5IChjYW5vbmljYWwgYW5jaG9yKSxDYW5vbmljYWwgYmFzZWxpbmUgYW5jaG9yIGZvciBub3JtYWxpemVkIHRva2VuLDIwMjEtMDktMDEsMjAyNS0xMS0wNCwwLCxBQ0NFUFRFRApLYW5wdXIsVmFyaXBhYWwgQVBNQyxWYXJpcGFhbCwiUnVsZSBBIChjaHJvbm9sb2dpY2FsIHRyYW5zaXRpb24sIG92ZXJsYXAgPD0gNWQpIixvdmVybGFwX2RheXM9NCA8PSA1IChjaHJvbm9sb2dpY2FsIG1pZ3JhdGlvbiksMjAyNS0xMS0wMSwyMDI2LTA0LTI2LDQsLEFDQ0VQVEVECkx1Y2tub3csQmFudGhhcmEsQmFudGhhcmEsSWRlbnRpdHkgKGNhbm9uaWNhbCBhbmNob3IpLENhbm9uaWNhbCBiYXNlbGluZSBhbmNob3IgZm9yIG5vcm1hbGl6ZWQgdG9rZW4sMjAyMi0wNS0xOCwyMDI1LTExLTA0LDAsLEFDQ0VQVEVECkx1Y2tub3csQmFudGhhcmEgQVBNQyxCYW50aGFyYSwiUnVsZSBBIChjaHJvbm9sb2dpY2FsIHRyYW5zaXRpb24sIG92ZXJsYXAgPD0gNWQpIixvdmVybGFwX2RheXM9NCA8PSA1IChjaHJvbm9sb2dpY2FsIG1pZ3JhdGlvbiksMjAyNS0xMS0wMSwyMDI2LTA5LTMwLDQsLEFDQ0VQVEVECkx1Y2tub3csTHVja25vdyxMdWNrbm93LElkZW50aXR5IChjYW5vbmljYWwgYW5jaG9yKSxDYW5vbmljYWwgYmFzZWxpbmUgYW5jaG9yIGZvciBub3JtYWxpemVkIHRva2VuLDIwMjEtMDktMDEsMjAyNS0xMS0wNCwwLCxBQ0NFUFRFRApMdWNrbm93LEx1Y2tub3cgQVBNQyxMdWNrbm93LCJSdWxlIEEgKGNocm9ub2xvZ2ljYWwgdHJhbnNpdGlvbiwgb3ZlcmxhcCA8PSA1ZCkiLG92ZXJsYXBfZGF5cz0zIDw9IDUgKGNocm9ub2xvZ2ljYWwgbWlncmF0aW9uKSwyMDI1LTExLTAyLDIwMjYtMDktMzAsMywsQUNDRVBURUQKVW5uYW8sQmFuZ2FybWF1LEJhbmdhcm1hdSxJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIGJhc2VsaW5lIGFuY2hvciBmb3Igbm9ybWFsaXplZCB0b2tlbiwyMDIxLTA5LTAxLDIwMjUtMTEtMDQsMCwsQUNDRVBURUQKVW5uYW8sQmFuZ2FybWF1IEFQTUMsQmFuZ2FybWF1LCJSdWxlIEEgKGNocm9ub2xvZ2ljYWwgdHJhbnNpdGlvbiwgb3ZlcmxhcCA8PSA1ZCkiLG92ZXJsYXBfZGF5cz00IDw9IDUgKGNocm9ub2xvZ2ljYWwgbWlncmF0aW9uKSwyMDI1LTExLTAxLDIwMjYtMDktMzAsNCwsQUNDRVBURUQKVW5uYW8sUHVyd2EsUHVyd2EsSWRlbnRpdHkgKGNhbm9uaWNhbCBhbmNob3IpLENhbm9uaWNhbCBiYXNlbGluZSBhbmNob3IgZm9yIG5vcm1hbGl6ZWQgdG9rZW4sMjAyMS0wOS0wMSwyMDI1LTA3LTMxLDAsLEFDQ0VQVEVEClVubmFvLFB1cndhIEFQTUMsUHVyd2EsIlJ1bGUgQSAoY2hyb25vbG9naWNhbCB0cmFuc2l0aW9uLCBvdmVybGFwIDw9IDVkKSIsb3ZlcmxhcF9kYXlzPTAgPD0gNSAoY2hyb25vbG9naWNhbCBtaWdyYXRpb24pLDIwMjYtMDItMjEsMjAyNi0wOS0zMCwwLCxBQ0NFUFRFRApVbm5hbyxVbm5hbyxVbm5hbyxJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIGJhc2VsaW5lIGFuY2hvciBmb3Igbm9ybWFsaXplZCB0b2tlbiwyMDIxLTA5LTAxLDIwMjUtMTEtMDQsMCwsQUNDRVBURUQKVW5uYW8sVW5uYW8gQVBNQyxVbm5hbywiUnVsZSBBIChjaHJvbm9sb2dpY2FsIHRyYW5zaXRpb24sIG92ZXJsYXAgPD0gNWQpIixvdmVybGFwX2RheXM9NCA8PSA1IChjaHJvbm9sb2dpY2FsIG1pZ3JhdGlvbiksMjAyNS0xMS0wMSwyMDI2LTA5LTMwLDQsLEFDQ0VQVEVECkthbnB1cixCYXJpcGFhbCBBUE1DLFZhcmlwYWFsLFJldmlldyBQZW5kaW5nIChjcm9zcy1waG9uZXRpYyBiL3YpLE5vcm1hbGl6ZWQgbWlzbWF0Y2ggKGJhcmlwYWFsIHZzIHZhcmlwYWFsKTsgY2FuZGlkYXRlIGZvciBtYW51YWwgdmVyaWZpY2F0aW9uLDIwMjYtMDMtMDEsMjAyNi0wOS0zMCwwLCxVTk1BUFBFRC9SRVZJRVcK",
    ref_dir / "market_alias_candidates_v21.csv": "ZGlzdHJpY3QsbWFya2V0X3JhdyxtYXJrZXRfY2Fub25pY2FsLHJ1bGUsZXZpZGVuY2UsZmlyc3RfZGF0ZSxsYXN0X2RhdGUsb3ZlcmxhcF9kYXlzLHByaWNlX2FncmVlbWVudCxzdGF0dXMsZXZpZGVuY2Vfa25vd25fZGF0ZSxhc3N1bWVkX2tub3duX2RhdGUsa25vd2xlZGdlX3Byb3ZlbmFuY2UKR2hhemlhYmFkLEdoYXppYWJhZCxHaGF6aWFiYWQsSWRlbnRpdHkgKGNhbm9uaWNhbCBhbmNob3IpLENhbm9uaWNhbCBiYXNlbGluZSBhbmNob3IgZm9yIG5vcm1hbGl6ZWQgdG9rZW4sMjAyMS0wOS0wMSwyMDI1LTExLTA0LDAsLEFDQ0VQVEVELCwyMDIxLTA5LTAxLFNUUlVDVFVSQUxfSURFTlRJVFlfQU5DSE9SCkdoYXppYWJhZCxHaGF6aWFiYWQgQVBNQyxHaGF6aWFiYWQsIkNhbmRpZGF0ZSBSdWxlIEEgKHZlcmlmaWVkIG1pZ3JhdGlvbjogZ2FwIDw9IDdkLCBvdmVybGFwIDItNGQsIDEwMCUgcHJpY2UgbWF0Y2gpIixnYXA9LTNkOyBvdmVybGFwPTRkOyAxMDMvMTAzIGtleXMgbWF0Y2ggMTAwLjAlLDIwMjUtMTEtMDEsMjAyNi0wOS0zMCw0LDEuMCxBQ0NFUFRFRCwsMjAyNS0xMS0wNSxBU1NVTUVEX09CU0VSVkFUSU9OX0hBTkRPVkVSCkdoYXppYWJhZCxIYXB1cixIYXB1cixJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIGJhc2VsaW5lIGFuY2hvciBmb3Igbm9ybWFsaXplZCB0b2tlbiwyMDIxLTA5LTAxLDIwMjUtMTEtMDUsMCwsQUNDRVBURUQsLDIwMjEtMDktMDEsU1RSVUNUVVJBTF9JREVOVElUWV9BTkNIT1IKR2hhemlhYmFkLEhhcHVyIEFQTUMsSGFwdXIsIkNhbmRpZGF0ZSBSdWxlIEEgKHZlcmlmaWVkIG1pZ3JhdGlvbjogZ2FwIDw9IDdkLCBvdmVybGFwIDItNGQsIDEwMCUgcHJpY2UgbWF0Y2gpIixnYXA9LTRkOyBvdmVybGFwPTRkOyA2MS82MSBrZXlzIG1hdGNoIDEwMC4wJSwyMDI1LTExLTAxLDIwMjYtMDktMzAsNCwxLjAsQUNDRVBURUQsLDIwMjUtMTEtMDUsQVNTVU1FRF9PQlNFUlZBVElPTl9IQU5ET1ZFUgpHaGF6aWFiYWQsTXVyYWRuYWdhcixNdXJhZG5hZ2FyLElkZW50aXR5IChjYW5vbmljYWwgYW5jaG9yKSxDYW5vbmljYWwgYmFzZWxpbmUgYW5jaG9yIGZvciBub3JtYWxpemVkIHRva2VuLDIwMjEtMDktMDEsMjAyNS0xMS0wNCwwLCxBQ0NFUFRFRCwsMjAyMS0wOS0wMSxTVFJVQ1RVUkFMX0lERU5USVRZX0FOQ0hPUgpHaGF6aWFiYWQsTXVyYWRuYWdhciBBUE1DLE11cmFkbmFnYXIsIkNhbmRpZGF0ZSBSdWxlIEEgKHZlcmlmaWVkIG1pZ3JhdGlvbjogZ2FwIDw9IDdkLCBvdmVybGFwIDItNGQsIDEwMCUgcHJpY2UgbWF0Y2gpIixnYXA9LTNkOyBvdmVybGFwPTRkOyAzNC8zNCBrZXlzIG1hdGNoIDEwMC4wJSwyMDI1LTExLTAxLDIwMjYtMDktMzAsNCwxLjAsQUNDRVBURUQsLDIwMjUtMTEtMDUsQVNTVU1FRF9PQlNFUlZBVElPTl9IQU5ET1ZFUgpHaGF6aWFiYWQsTm9pZGEsTm9pZGEsSWRlbnRpdHkgKGNhbm9uaWNhbCBhbmNob3IpLENhbm9uaWNhbCBiYXNlbGluZSBhbmNob3IgZm9yIG5vcm1hbGl6ZWQgdG9rZW4sMjAyMS0wOS0wMiwyMDI1LTExLTAzLDAsLEFDQ0VQVEVELCwyMDIxLTA5LTAxLFNUUlVDVFVSQUxfSURFTlRJVFlfQU5DSE9SCkdoYXppYWJhZCxOb2lkYSBBUE1DLE5vaWRhLCJDYW5kaWRhdGUgUnVsZSBBICh2ZXJpZmllZCBtaWdyYXRpb246IGdhcCA8PSA3ZCwgb3ZlcmxhcCAyLTRkLCAxMDAlIHByaWNlIG1hdGNoKSIsZ2FwPS0yZDsgb3ZlcmxhcD0yZDsgMzkvMzkga2V5cyBtYXRjaCAxMDAuMCUsMjAyNS0xMS0wMSwyMDI2LTA5LTMwLDIsMS4wLEFDQ0VQVEVELCwyMDI1LTExLTA1LEFTU1VNRURfT0JTRVJWQVRJT05fSEFORE9WRVIKSmhhbnNpLEJhcnV3YXNhZ2FyLEJhcnV3YXNhZ2FyLElkZW50aXR5IChjYW5vbmljYWwgYW5jaG9yKSxDYW5vbmljYWwgYmFzZWxpbmUgYW5jaG9yIGZvciBub3JtYWxpemVkIHRva2VuLDIwMjEtMDktMDIsMjAyNS0xMS0wNSwwLCxBQ0NFUFRFRCwsMjAyMS0wOS0wMSxTVFJVQ1RVUkFMX0lERU5USVRZX0FOQ0hPUgpKaGFuc2ksQmFydXdhc2FnYXIgQVBNQyxCYXJ1d2FzYWdhciwiQ2FuZGlkYXRlIFJ1bGUgQSAodmVyaWZpZWQgbWlncmF0aW9uOiBnYXAgPD0gN2QsIG92ZXJsYXAgMi00ZCwgMTAwJSBwcmljZSBtYXRjaCkiLGdhcD0tMmQ7IG92ZXJsYXA9M2Q7IDUvNSBrZXlzIG1hdGNoIDEwMC4wJSwyMDI1LTExLTAzLDIwMjYtMDktMzAsMywxLjAsQUNDRVBURUQsLDIwMjUtMTEtMDUsQVNTVU1FRF9PQlNFUlZBVElPTl9IQU5ET1ZFUgpKaGFuc2ksQ2hpcmdhb24sQ2hpcmdhb24sSWRlbnRpdHkgKGNhbm9uaWNhbCBhbmNob3IpLENhbm9uaWNhbCBiYXNlbGluZSBhbmNob3IgZm9yIG5vcm1hbGl6ZWQgdG9rZW4sMjAyMS0wOS0wMSwyMDI1LTExLTA0LDAsLEFDQ0VQVEVELCwyMDIxLTA5LTAxLFNUUlVDVFVSQUxfSURFTlRJVFlfQU5DSE9SCkpoYW5zaSxDaGlyZ2FvbiBBUE1DLENoaXJnYW9uLCJDYW5kaWRhdGUgUnVsZSBBICh2ZXJpZmllZCBtaWdyYXRpb246IGdhcCA8PSA3ZCwgb3ZlcmxhcCAyLTRkLCAxMDAlIHByaWNlIG1hdGNoKSIsZ2FwPS0yZDsgb3ZlcmxhcD0yZDsgNi82IGtleXMgbWF0Y2ggMTAwLjAlLDIwMjUtMTEtMDIsMjAyNi0wOS0zMCwyLDEuMCxBQ0NFUFRFRCwsMjAyNS0xMS0wNSxBU1NVTUVEX09CU0VSVkFUSU9OX0hBTkRPVkVSCkpoYW5zaSxHdXJ1c2FyYWksR3VydXNhcmFpLElkZW50aXR5IChjYW5vbmljYWwgYW5jaG9yKSxDYW5vbmljYWwgYmFzZWxpbmUgYW5jaG9yIGZvciBub3JtYWxpemVkIHRva2VuLDIwMjEtMDktMDEsMjAyNS0xMS0wNCwwLCxBQ0NFUFRFRCwsMjAyMS0wOS0wMSxTVFJVQ1RVUkFMX0lERU5USVRZX0FOQ0hPUgpKaGFuc2ksR3VydXNhcmFpIEFQTUMsR3VydXNhcmFpLCJDYW5kaWRhdGUgUnVsZSBBICh2ZXJpZmllZCBtaWdyYXRpb246IGdhcCA8PSA3ZCwgb3ZlcmxhcCAyLTRkLCAxMDAlIHByaWNlIG1hdGNoKSIsZ2FwPS0zZDsgb3ZlcmxhcD00ZDsgMjgvMjgga2V5cyBtYXRjaCAxMDAuMCUsMjAyNS0xMS0wMSwyMDI2LTA5LTMwLDQsMS4wLEFDQ0VQVEVELCwyMDI1LTExLTA1LEFTU1VNRURfT0JTRVJWQVRJT05fSEFORE9WRVIKSmhhbnNpLEpoYW5zaSxKaGFuc2ksSWRlbnRpdHkgKGNhbm9uaWNhbCBhbmNob3IpLENhbm9uaWNhbCBiYXNlbGluZSBhbmNob3IgZm9yIG5vcm1hbGl6ZWQgdG9rZW4sMjAyMS0wOS0wMSwyMDI1LTExLTA0LDAsLEFDQ0VQVEVELCwyMDIxLTA5LTAxLFNUUlVDVFVSQUxfSURFTlRJVFlfQU5DSE9SCkpoYW5zaSxKaGFuc2kgQVBNQyxKaGFuc2ksIkNhbmRpZGF0ZSBSdWxlIEEgKHZlcmlmaWVkIG1pZ3JhdGlvbjogZ2FwIDw9IDdkLCBvdmVybGFwIDItNGQsIDEwMCUgcHJpY2UgbWF0Y2gpIixnYXA9LTNkOyBvdmVybGFwPTRkOyA0OS80OSBrZXlzIG1hdGNoIDEwMC4wJSwyMDI1LTExLTAxLDIwMjYtMDktMzAsNCwxLjAsQUNDRVBURUQsLDIwMjUtMTEtMDUsQVNTVU1FRF9PQlNFUlZBVElPTl9IQU5ET1ZFUgpKaGFuc2ksSmhhbnNpIChHcmFpbiksSmhhbnNpIChHcmFpbiksSWRlbnRpdHkgKGNhbm9uaWNhbCBhbmNob3IpLENhbm9uaWNhbCBiYXNlbGluZSBhbmNob3IgZm9yIG5vcm1hbGl6ZWQgdG9rZW4sMjAyNC0wNS0yNSwyMDI1LTExLTAzLDAsLEFDQ0VQVEVELCwyMDIxLTA5LTAxLFNUUlVDVFVSQUxfSURFTlRJVFlfQU5DSE9SCkpoYW5zaSxKaGFuc2kgKEdyYWluKSBBUE1DLEpoYW5zaSAoR3JhaW4pLCJDYW5kaWRhdGUgUnVsZSBBICh2ZXJpZmllZCBtaWdyYXRpb246IGdhcCA8PSA3ZCwgb3ZlcmxhcCAyLTRkLCAxMDAlIHByaWNlIG1hdGNoKSIsZ2FwPS0xZDsgb3ZlcmxhcD0yZDsgNS81IGtleXMgbWF0Y2ggMTAwLjAlLDIwMjUtMTEtMDIsMjAyNi0wMS0xNCwyLDEuMCxBQ0NFUFRFRCwsMjAyNS0xMS0wNSxBU1NVTUVEX09CU0VSVkFUSU9OX0hBTkRPVkVSCkpoYW5zaSxKaGFuc2kgR3JhaW4gQVBNQyxKaGFuc2kgKEdyYWluKSxDYW5kaWRhdGUgUnVsZSBCICh0cmFuc2l0aXZlIGNvbmNvcmRhbmNlIHZpYSBKaGFuc2kgKEdyYWluKSBBUE1DKSwxMGQgb3ZlcmxhcCB3aXRoIEpoYW5zaSAoR3JhaW4pIEFQTUMgKDIwMjYtMDEtMDEuLjE0KTsgMjMvMjMga2V5cyBhZ3JlZSAxMDAlOyB0cmFuc2l0aXZlIGFuY2hvciBKaGFuc2kgKEdyYWluKSwyMDI2LTAxLTAxLDIwMjYtMDEtMTQsMTAsMS4wLEFDQ0VQVEVELCwyMDI2LTAxLTE0LEFTU1VNRURfT0JTRVJWQVRJT05fSEFORE9WRVIKSmhhbnNpLE1hdXJhbmlwdXIsTWF1cmFuaXB1cixJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIGJhc2VsaW5lIGFuY2hvciBmb3Igbm9ybWFsaXplZCB0b2tlbiwyMDIxLTA5LTAxLDIwMjUtMTEtMDQsMCwsQUNDRVBURUQsLDIwMjEtMDktMDEsU1RSVUNUVVJBTF9JREVOVElUWV9BTkNIT1IKSmhhbnNpLE1hdXJhbmlwdXIgQVBNQyxNYXVyYW5pcHVyLCJDYW5kaWRhdGUgUnVsZSBBICh2ZXJpZmllZCBtaWdyYXRpb246IGdhcCA8PSA3ZCwgb3ZlcmxhcCAyLTRkLCAxMDAlIHByaWNlIG1hdGNoKSIsZ2FwPS0zZDsgb3ZlcmxhcD00ZDsgNDMvNDMga2V5cyBtYXRjaCAxMDAuMCUsMjAyNS0xMS0wMSwyMDI2LTA5LTMwLDQsMS4wLEFDQ0VQVEVELCwyMDI1LTExLTA1LEFTU1VNRURfT0JTRVJWQVRJT05fSEFORE9WRVIKSmhhbnNpLE1vdGgsTW90aCxJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIGJhc2VsaW5lIGFuY2hvciBmb3Igbm9ybWFsaXplZCB0b2tlbiwyMDIxLTA5LTAxLDIwMjUtMTEtMDQsMCwsQUNDRVBURUQsLDIwMjEtMDktMDEsU1RSVUNUVVJBTF9JREVOVElUWV9BTkNIT1IKSmhhbnNpLE1vdGggQVBNQyxNb3RoLCJDYW5kaWRhdGUgUnVsZSBBICh2ZXJpZmllZCBtaWdyYXRpb246IGdhcCA8PSA3ZCwgb3ZlcmxhcCAyLTRkLCAxMDAlIHByaWNlIG1hdGNoKSIsZ2FwPS0zZDsgb3ZlcmxhcD00ZDsgMzAvMzAga2V5cyBtYXRjaCAxMDAuMCUsMjAyNS0xMS0wMSwyMDI2LTA5LTMwLDQsMS4wLEFDQ0VQVEVELCwyMDI1LTExLTA1LEFTU1VNRURfT0JTRVJWQVRJT05fSEFORE9WRVIKS2FucHVyLEJhcmlwYWFsIEFQTUMsQmFyaXBhYWwgQVBNQyxJZGVudGl0eSAodW5tZXJnZWQgbWFya2V0IHlhcmQpLFVuaXF1ZSBtYXJrZXQgZm9yIG5vcm1hbGl6ZWQgdG9rZW4gaW4gZGlzdHJpY3QsMjAyNi0wMy0wMSwyMDI2LTA5LTMwLDAsLEFDQ0VQVEVELCwyMDI2LTAzLTAxLFNUUlVDVFVSQUxfSURFTlRJVFlfQU5DSE9SCkthbnB1cixDaG91YmVwdXIsQ2hvdWJlcHVyLElkZW50aXR5IChjYW5vbmljYWwgYW5jaG9yKSxDYW5vbmljYWwgYmFzZWxpbmUgYW5jaG9yIGZvciBub3JtYWxpemVkIHRva2VuLDIwMjEtMDktMDEsMjAyNS0xMS0wNCwwLCxBQ0NFUFRFRCwsMjAyMS0wOS0wMSxTVFJVQ1RVUkFMX0lERU5USVRZX0FOQ0hPUgpLYW5wdXIsQ2hvdWJlcHVyIEFQTUMsQ2hvdWJlcHVyLCJDYW5kaWRhdGUgUnVsZSBBICh2ZXJpZmllZCBtaWdyYXRpb246IGdhcCA8PSA3ZCwgb3ZlcmxhcCAyLTRkLCAxMDAlIHByaWNlIG1hdGNoKSIsZ2FwPS0zZDsgb3ZlcmxhcD0zZDsgNjYvNjYga2V5cyBtYXRjaCAxMDAuMCUsMjAyNS0xMS0wMSwyMDI2LTA5LTMwLDMsMS4wLEFDQ0VQVEVELCwyMDI1LTExLTA1LEFTU1VNRURfT0JTRVJWQVRJT05fSEFORE9WRVIKS2FucHVyLEthbnB1cihHcmFpbiksS2FucHVyKEdyYWluKSxJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIGJhc2VsaW5lIGFuY2hvciBmb3Igbm9ybWFsaXplZCB0b2tlbiwyMDIxLTA5LTAxLDIwMjUtMTEtMDQsMCwsQUNDRVBURUQsLDIwMjEtMDktMDEsU1RSVUNUVVJBTF9JREVOVElUWV9BTkNIT1IKS2FucHVyLEthbnB1cihHcmFpbikgQVBNQyxLYW5wdXIoR3JhaW4pLCJDYW5kaWRhdGUgUnVsZSBBICh2ZXJpZmllZCBtaWdyYXRpb246IGdhcCA8PSA3ZCwgb3ZlcmxhcCAyLTRkLCAxMDAlIHByaWNlIG1hdGNoKSIsZ2FwPS0zZDsgb3ZlcmxhcD0zZDsgNjMvNjMga2V5cyBtYXRjaCAxMDAuMCUsMjAyNS0xMS0wMSwyMDI2LTA5LTMwLDMsMS4wLEFDQ0VQVEVELCwyMDI1LTExLTA1LEFTU1VNRURfT0JTRVJWQVRJT05fSEFORE9WRVIKS2FucHVyLFV0dGFyaXB1cmEsVXR0YXJpcHVyYSxJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIGJhc2VsaW5lIGFuY2hvciBmb3Igbm9ybWFsaXplZCB0b2tlbiwyMDIxLTEwLTI0LDIwMjUtMTEtMDQsMCwsQUNDRVBURUQsLDIwMjEtMDktMDEsU1RSVUNUVVJBTF9JREVOVElUWV9BTkNIT1IKS2FucHVyLFV0dGFyaXB1cmEgQVBNQyxVdHRhcmlwdXJhLCJDYW5kaWRhdGUgUnVsZSBBICh2ZXJpZmllZCBtaWdyYXRpb246IGdhcCA8PSA3ZCwgb3ZlcmxhcCAyLTRkLCAxMDAlIHByaWNlIG1hdGNoKSIsZ2FwPS0zZDsgb3ZlcmxhcD0zZDsgNjYvNjYga2V5cyBtYXRjaCAxMDAuMCUsMjAyNS0xMS0wMSwyMDI2LTA5LTMwLDMsMS4wLEFDQ0VQVEVELCwyMDI1LTExLTA1LEFTU1VNRURfT0JTRVJWQVRJT05fSEFORE9WRVIKS2FucHVyLFZhcmlwYWFsLFZhcmlwYWFsLElkZW50aXR5IChjYW5vbmljYWwgYW5jaG9yKSxDYW5vbmljYWwgYmFzZWxpbmUgYW5jaG9yIGZvciBub3JtYWxpemVkIHRva2VuLDIwMjEtMDktMDEsMjAyNS0xMS0wNCwwLCxBQ0NFUFRFRCwsMjAyMS0wOS0wMSxTVFJVQ1RVUkFMX0lERU5USVRZX0FOQ0hPUgpLYW5wdXIsVmFyaXBhYWwgQVBNQyxWYXJpcGFhbCwiQ2FuZGlkYXRlIFJ1bGUgQSAodmVyaWZpZWQgbWlncmF0aW9uOiBnYXAgPD0gN2QsIG92ZXJsYXAgMi00ZCwgMTAwJSBwcmljZSBtYXRjaCkiLGdhcD0tM2Q7IG92ZXJsYXA9M2Q7IDQ3LzQ3IGtleXMgbWF0Y2ggMTAwLjAlLDIwMjUtMTEtMDEsMjAyNi0wNC0yNiwzLDEuMCxBQ0NFUFRFRCwsMjAyNS0xMS0wNSxBU1NVTUVEX09CU0VSVkFUSU9OX0hBTkRPVkVSCkx1Y2tub3csQmFudGhhcmEsQmFudGhhcmEsSWRlbnRpdHkgKGNhbm9uaWNhbCBhbmNob3IpLENhbm9uaWNhbCBiYXNlbGluZSBhbmNob3IgZm9yIG5vcm1hbGl6ZWQgdG9rZW4sMjAyMi0wNS0xOCwyMDI1LTExLTA0LDAsLEFDQ0VQVEVELCwyMDIxLTA5LTAxLFNUUlVDVFVSQUxfSURFTlRJVFlfQU5DSE9SCkx1Y2tub3csQmFudGhhcmEgQVBNQyxCYW50aGFyYSwiQ2FuZGlkYXRlIFJ1bGUgQSAodmVyaWZpZWQgbWlncmF0aW9uOiBnYXAgPD0gN2QsIG92ZXJsYXAgMi00ZCwgMTAwJSBwcmljZSBtYXRjaCkiLGdhcD0tM2Q7IG92ZXJsYXA9NGQ7IDgvOCBrZXlzIG1hdGNoIDEwMC4wJSwyMDI1LTExLTAxLDIwMjYtMDktMzAsNCwxLjAsQUNDRVBURUQsLDIwMjUtMTEtMDUsQVNTVU1FRF9PQlNFUlZBVElPTl9IQU5ET1ZFUgpMdWNrbm93LEx1Y2tub3csTHVja25vdyxJZGVudGl0eSAoY2Fub25pY2FsIGFuY2hvciksQ2Fub25pY2FsIGJhc2VsaW5lIGFuY2hvciBmb3Igbm9ybWFsaXplZCB0b2tlbiwyMDIxLTA5LTAxLDIwMjUtMTEtMDQsMCwsQUNDRVBURUQsLDIwMjEtMDktMDEsU1RSVUNUVVJBTF9JREVOVElUWV9BTkNIT1IKTHVja25vdyxMdWNrbm93IEFQTUMsTHVja25vdywiQ2FuZGlkYXRlIFJ1bGUgQSAodmVyaWZpZWQgbWlncmF0aW9uOiBnYXAgPD0gN2QsIG92ZXJsYXAgMi00ZCwgMTAwJSBwcmljZSBtYXRjaCkiLGdhcD0tMmQ7IG92ZXJsYXA9M2Q7IDY0LzY0IGtleXMgbWF0Y2ggMTAwLjAlLDIwMjUtMTEtMDIsMjAyNi0wOS0zMCwzLDEuMCxBQ0NFUFRFRCwsMjAyNS0xMS0wNSxBU1NVTUVEX09CU0VSVkFUSU9OX0hBTkRPVkVSClVubmFvLEJhbmdhcm1hdSxCYW5nYXJtYXUsSWRlbnRpdHkgKGNhbm9uaWNhbCBhbmNob3IpLENhbm9uaWNhbCBiYXNlbGluZSBhbmNob3IgZm9yIG5vcm1hbGl6ZWQgdG9rZW4sMjAyMS0wOS0wMSwyMDI1LTExLTA0LDAsLEFDQ0VQVEVELCwyMDIxLTA5LTAxLFNUUlVDVFVSQUxfSURFTlRJVFlfQU5DSE9SClVubmFvLEJhbmdhcm1hdSBBUE1DLEJhbmdhcm1hdSwiQ2FuZGlkYXRlIFJ1bGUgQSAodmVyaWZpZWQgbWlncmF0aW9uOiBnYXAgPD0gN2QsIG92ZXJsYXAgMi00ZCwgMTAwJSBwcmljZSBtYXRjaCkiLGdhcD0tM2Q7IG92ZXJsYXA9NGQ7IDg4Lzg4IGtleXMgbWF0Y2ggMTAwLjAlLDIwMjUtMTEtMDEsMjAyNi0wOS0zMCw0LDEuMCxBQ0NFUFRFRCwsMjAyNS0xMS0wNSxBU1NVTUVEX09CU0VSVkFUSU9OX0hBTkRPVkVSClVubmFvLFB1cndhLFB1cndhLElkZW50aXR5IChjYW5vbmljYWwgYW5jaG9yKSxDYW5vbmljYWwgYmFzZWxpbmUgYW5jaG9yIGZvciBub3JtYWxpemVkIHRva2VuLDIwMjEtMDktMDEsMjAyNS0wNy0zMSwwLCxBQ0NFUFRFRCwsMjAyMS0wOS0wMSxTVFJVQ1RVUkFMX0lERU5USVRZX0FOQ0hPUgpVbm5hbyxQdXJ3YSBBUE1DLFB1cndhLENhbmRpZGF0ZSBSdWxlIEEgRmFpbGVkIChoaWF0dXM9MjA1ZCA+IDE0ZCBsaW1pdDsgMCBvdmVybGFwIGtleXMpLCJPbGRlciBQdXJ3YSBlbmRlZCAyMDI1LTA3LTMxOyBQdXJ3YSBBUE1DIHN0YXJ0ZWQgMjAyNi0wMi0yMSAoZ2FwPTIwNWQsIG92ZXJsYXA9MGQsIGtleXM9MCk7IHVudmVyaWZpZWQgdHJhbnNpdGlvbiBjb25jb3JkYW5jZSIsMjAyNi0wMi0yMSwyMDI2LTA5LTMwLDAsLFVOTUFQUEVEL1JFVklFVywsMjAyNi0wMi0yMSxBU1NVTUVEX09CU0VSVkFUSU9OX0hBTkRPVkVSClVubmFvLFVubmFvLFVubmFvLElkZW50aXR5IChjYW5vbmljYWwgYW5jaG9yKSxDYW5vbmljYWwgYmFzZWxpbmUgYW5jaG9yIGZvciBub3JtYWxpemVkIHRva2VuLDIwMjEtMDktMDEsMjAyNS0xMS0wNCwwLCxBQ0NFUFRFRCwsMjAyMS0wOS0wMSxTVFJVQ1RVUkFMX0lERU5USVRZX0FOQ0hPUgpVbm5hbyxVbm5hbyBBUE1DLFVubmFvLCJDYW5kaWRhdGUgUnVsZSBBICh2ZXJpZmllZCBtaWdyYXRpb246IGdhcCA8PSA3ZCwgb3ZlcmxhcCAyLTRkLCAxMDAlIHByaWNlIG1hdGNoKSIsZ2FwPS0zZDsgb3ZlcmxhcD0zZDsgMjcvMjcga2V5cyBtYXRjaCAxMDAuMCUsMjAyNS0xMS0wMSwyMDI2LTA5LTMwLDMsMS4wLEFDQ0VQVEVELCwyMDI1LTExLTA1LEFTU1VNRURfT0JTRVJWQVRJT05fSEFORE9WRVIKS2FucHVyLEJhcmlwYWFsIEFQTUMsVmFyaXBhYWwsQ2FuZGlkYXRlIFJ1bGUgQiBGYWlsZWQgKGNyb3NzLXBob25ldGljIGIvdjsgMjAgY29uZmxpY3Rpbmcga2V5cyksNTJkIGNvLXJlcG9ydGluZyAoMjAyNi0wMy0wMS4uMDQtMjYpIHdpdGggVmFyaXBhYWwgQVBNQzsgNDQyLzQ2MiBtb2RhbCBtYXRjaCAoOTUuNjclKSBidXQgMjAga2V5cyBjb25mbGljdDsgbm9ybWFsaXplZCBtaXNtYXRjaCwyMDI2LTAzLTAxLDIwMjYtMDktMzAsNTIsMC45NTY3LFVOTUFQUEVEL1JFVklFVywsMjAyNi0wNC0yNixBU1NVTUVEX09CU0VSVkFUSU9OX0hBTkRPVkVSCg=="
}

print("\n=== DECODING & VERIFYING EMBEDDED MODULE PAYLOADS ===")
for target_path, b64_str in b64_payloads.items():
    decoded_bytes = base64.b64decode(b64_str.encode("ascii"))
    target_path.write_bytes(decoded_bytes)
    written_bytes = target_path.read_bytes()
    assert written_bytes == decoded_bytes, f"Byte mismatch writing {target_path}"
    print(f"  OK: {target_path.relative_to(working_root)} ({len(written_bytes):,} bytes, sha256={hashlib.sha256(written_bytes).hexdigest()[:16]}...)")

if str(working_root) not in sys.path:
    sys.path.insert(0, str(working_root))

print("\n=== DISCOVERING INPUT DATASET ===")
KAGGLE_INPUT_DIR = None
potential_roots = [
    Path("/kaggle/input/mandi-prediction-audit-003b"),
    Path("/kaggle/input/mandi-prediction-audit-003"),
    Path("/kaggle/input"),
    Path(".")
]
for p in potential_roots:
    if (p / "5 year dataset").exists():
        KAGGLE_INPUT_DIR = str(p)
        print(f"Found dataset root at: {KAGGLE_INPUT_DIR}")
        break

if KAGGLE_INPUT_DIR is None:
    print("WARNING: '5 year dataset' not found in standard paths; checking working dir...")
    KAGGLE_INPUT_DIR = str(working_root)

raw_csv_files = sorted(glob.glob(os.path.join(KAGGLE_INPUT_DIR, "5 year dataset/*.csv")))
if not raw_csv_files:
    raw_csv_files = sorted(glob.glob("5 year dataset/*.csv"))
print(f"Found {len(raw_csv_files)} raw CSV files.")
for f in raw_csv_files:
    print(f"  RAW: {os.path.basename(f)} ({os.path.getsize(f):,} bytes)")


In [ ]:
# Cell 2: Phase 1: Fail-Closed Reproduction of Legacy Panel v2
print("\n=== PHASE 1: FAIL-CLOSED REPRODUCTION OF LEGACY PANEL V2 ===")
EXPECTED_PANEL_V2_SHA256 = "851e718468d6b8c9fd07944535a462338e71aa105af25339fe9438b697088f31"

from src.data_pipeline import MandiDataPipeline, compute_file_sha256
from src.identity_comparison import extract_manifest_dispositions

t0 = time.time()
v2_panel_path = working_root / "mandi_5year_validated_v2.csv"
v2_manifest_path = working_root / "processing_manifest_v2.json"

df_alias_all = pd.read_csv(ref_dir / "market_aliases.csv")
df_alias_accepted = df_alias_all[df_alias_all["status"] == "ACCEPTED"].copy()
print(f"Loaded {len(df_alias_accepted)} accepted legacy alias entries.")

pipeline_v2 = MandiDataPipeline(
    input_dir=os.path.join(KAGGLE_INPUT_DIR, "5 year dataset"),
    output_dir=str(working_root),
    output_filename="mandi_5year_validated_v2.csv",
    manifest_filename="processing_manifest_v2.json",
    min_series_records=20,
    exclude_ambiguous_same_grade=True,
    pipeline_version="v2",
    alias_df=df_alias_accepted
)

df_panel_v2, manifest_v2, out_csv_v2, out_manifest_v2 = pipeline_v2.run(raw_csv_files)
elapsed_v2 = time.time() - t0
print(f"Panel v2 generated in {elapsed_v2:.2f} seconds.")

v2_sha256 = compute_file_sha256(str(v2_panel_path))
print(f"Computed Panel v2 SHA-256: {v2_sha256}")
print(f"Expected Panel v2 SHA-256: {EXPECTED_PANEL_V2_SHA256}")

# Strict Fail-Closed Assertion
if v2_sha256 != EXPECTED_PANEL_V2_SHA256:
    print(f"FATAL: Panel v2 hash mismatch! {v2_sha256} != {EXPECTED_PANEL_V2_SHA256}")
    raise RuntimeError(f"Panel v2 reproduction failed! Bit-level reproducibility violated.")

print("PASS: Panel v2 exactly matches known immutable reference hash.")

# Extract manifest dispositions using real nested schema without measurement fallbacks
dispositions_v2 = extract_manifest_dispositions(manifest_v2)
print("\nPanel v2 Dispositions & Exact Conservation Check:")
print(f"  Raw Rows In:            {dispositions_v2['raw_rows_in']:,}")
print(f"  Valid Parsed Rows:      {dispositions_v2['valid_parsed_rows']:,}")
print(f"  Invalid Dropped Rows:   {dispositions_v2['invalid_dropped_rows']:,}")
print(f"  Duplicates Deduplicated:{dispositions_v2['duplicates_deduplicated']:,}")
print(f"  Conflicts Quarantined:  {dispositions_v2['conflicts_quarantined']:,}")
print(f"  Low-Obs Dropped:        {dispositions_v2['low_obs_dropped']:,}")
print(f"  Final Clean Rows:       {dispositions_v2['accepted_rows']:,}")
print(f"  Row Conservation:       {dispositions_v2['row_conservation_verified']}")


In [ ]:
# Cell 3: Phase 2: Candidate Panel v2.1 Generation & Comparison
print("\n=== PHASE 2: CANDIDATE PANEL V2.1 GENERATION & COMPARISON ===")

from src.market_identity import load_market_aliases, validate_alias_table
from src.identity_comparison import compare_panel_dispositions

alias_cand_path = ref_dir / "market_alias_candidates_v21.csv"
cand_alias_df = load_market_aliases(alias_cand_path)
is_valid, validation_issues = validate_alias_table(cand_alias_df)
print(f"Candidate Alias Table Validation: valid={is_valid}, issues={validation_issues}")
assert is_valid, f"Candidate alias table failed validation: {validation_issues}"

cand_alias_accepted = cand_alias_df[cand_alias_df["status"] == "ACCEPTED"].copy()
print(f"Loaded {len(cand_alias_accepted)} accepted candidate v2.1 aliases.")

t0 = time.time()
v21_panel_path = working_root / "mandi_5year_validated_v21.csv"
v21_manifest_path = working_root / "processing_manifest_v21.json"

pipeline_v21 = MandiDataPipeline(
    input_dir=os.path.join(KAGGLE_INPUT_DIR, "5 year dataset"),
    output_dir=str(working_root),
    output_filename="mandi_5year_validated_v21.csv",
    manifest_filename="processing_manifest_v21.json",
    min_series_records=20,
    exclude_ambiguous_same_grade=True,
    pipeline_version="v2",
    alias_df=cand_alias_accepted
)

df_panel_v21, manifest_v21, out_csv_v21, out_manifest_v21 = pipeline_v21.run(raw_csv_files)
elapsed_v21 = time.time() - t0
v21_sha256 = compute_file_sha256(str(v21_panel_path))
print(f"Candidate Panel v2.1 generated in {elapsed_v21:.2f} seconds.")
print(f"Computed Candidate Panel v2.1 SHA-256: {v21_sha256}")

# Compute side-by-side comparison using shared comparison helper
comp = compare_panel_dispositions(manifest_v2, manifest_v21, df_panel_v2, df_panel_v21)

print("\n--- District Comparison: Panel v2 vs Candidate v2.1 ---")
print(comp["district_comparison"].to_string(index=False))

print("\n--- Overall Disposition Deltas (Candidate v2.1 - Legacy v2) ---")
for k, v in comp["overall_delta"].items():
    print(f"  {k}: {v}")

# Focal Market Identity Audits
print("\n--- Focal Market Identity Audits ---")

# Purwa Audit
purwa_v2 = df_panel_v2[df_panel_v2["Market_Raw"].str.contains("Purwa", na=False)]
purwa_v21 = df_panel_v21[df_panel_v21["Market_Raw"].str.contains("Purwa", na=False)]
print(f"Purwa Rows (Raw): v2={len(purwa_v2)}, v2.1={len(purwa_v21)}")
print(f"  v2 Canonical Market values: {purwa_v2['Market'].unique()}")
print(f"  v2.1 Canonical Market values: {purwa_v21['Market'].unique()}")

# Jhansi Grain Audit
jhansi_grain_v2 = df_panel_v2[df_panel_v2["Market_Raw"].str.contains("Jhansi.*Grain", na=False)]
jhansi_grain_v21 = df_panel_v21[df_panel_v21["Market_Raw"].str.contains("Jhansi.*Grain", na=False)]
print(f"Jhansi Grain Rows (Raw): v2={len(jhansi_grain_v2)}, v2.1={len(jhansi_grain_v21)}")
print(f"  v2 Canonical Market values: {jhansi_grain_v2['Market'].unique()}")
print(f"  v2.1 Canonical Market values: {jhansi_grain_v21['Market'].unique()}")

# Baripaal Audit
baripaal_v2 = df_panel_v2[df_panel_v2["Market_Raw"].str.contains("Baripaal", na=False)]
baripaal_v21 = df_panel_v21[df_panel_v21["Market_Raw"].str.contains("Baripaal", na=False)]
print(f"Baripaal Rows (Raw): v2={len(baripaal_v2)}, v2.1={len(baripaal_v21)}")
print(f"  v2 Canonical Market values: {baripaal_v2['Market'].unique()}")
print(f"  v2.1 Canonical Market values: {baripaal_v21['Market'].unique()}")

comp["district_comparison"].to_csv(out_dir / "identity_comparison_v2_vs_v21.csv", index=False)
print("\nExported identity_comparison_v2_vs_v21.csv.")


In [ ]:
# Cell 4: Phase 3: Forensic Reclassification & Causal Sensitivity Verification
print("\n=== PHASE 3: FORENSIC RECLASSIFICATION & CAUSAL SENSITIVITY VERIFICATION ===")

from src.identity_comparison import (
    classify_excel_against_panel,
    evaluate_causal_train_eligibility_sensitivity,
    verify_future_mutation_invariance
)

# 1. Discover in-window Excel reports
excel_candidates = sorted(glob.glob(os.path.join(KAGGLE_INPUT_DIR, "*Potato*.xlsx"))) +                    sorted(glob.glob(os.path.join(KAGGLE_INPUT_DIR, "*Wheat*.xlsx"))) +                    sorted(glob.glob(os.path.join(KAGGLE_INPUT_DIR, "**/*Potato*.xlsx"), recursive=True))

excel_dfs = []
for p in set(excel_candidates):
    try:
        df_ex = pd.read_excel(p)
        excel_dfs.append(df_ex)
    except Exception as e:
        print(f"Note: Could not read {p}: {e}")

if excel_dfs:
    df_excel_all = pd.concat(excel_dfs, ignore_index=True)
    # Filter in-window dates (2021-09-01 to 2026-06-30)
    df_excel_all["Arrival_Date_dt"] = pd.to_datetime(df_excel_all["Arrival_Date"], errors="coerce")
    df_excel_in_window = df_excel_all[
        (df_excel_all["Arrival_Date_dt"] >= "2021-09-01") &
        (df_excel_all["Arrival_Date_dt"] <= "2026-06-30")
    ].copy()
    print(f"Discovered {len(df_excel_in_window)} in-window Excel observations.")
else:
    print("No Excel files discovered in input; creating mock verification set...")
    df_excel_in_window = pd.DataFrame()

# Perform dynamic reclassification on both v2 and v2.1
if not df_excel_in_window.empty:
    print("\nReclassifying Excel observations against Panel v2...")
    clf_v2 = classify_excel_against_panel(df_excel_in_window, df_panel_v2)
    print(f"  v2 Matched Exact:     {clf_v2['matched_exact']:,}")
    print(f"  v2 Matched Different: {clf_v2['matched_different']:,}")
    print(f"  v2 Unmatched Total:   {clf_v2['unmatched_total']:,}")
    print(f"  v2 Categories:        {clf_v2['categories']}")

    print("\nReclassifying Excel observations against Candidate Panel v2.1...")
    clf_v21 = classify_excel_against_panel(df_excel_in_window, df_panel_v21)
    print(f"  v2.1 Matched Exact:     {clf_v21['matched_exact']:,}")
    print(f"  v2.1 Matched Different: {clf_v21['matched_different']:,}")
    print(f"  v2.1 Unmatched Total:   {clf_v21['unmatched_total']:,}")
    print(f"  v2.1 Categories:        {clf_v21['categories']}")

    unmatched_summary = {
        "panel_v2_sha256": v2_sha256,
        "panel_v21_sha256": v21_sha256,
        "total_in_window_excel_rows": len(df_excel_in_window),
        "v2_matched_exact": clf_v2["matched_exact"],
        "v2_matched_different": clf_v2["matched_different"],
        "v2_unmatched_categories": clf_v2["categories"],
        "v21_matched_exact": clf_v21["matched_exact"],
        "v21_matched_different": clf_v21["matched_different"],
        "v21_unmatched_categories": clf_v21["categories"],
        "category_conservation_check": clf_v21["category_conservation_check"]
    }
else:
    print("Excel input unavailable; recording NOT_IMPLEMENTED for execution acceptance.")
    unmatched_summary = {
        "panel_v2_sha256": v2_sha256,
        "panel_v21_sha256": v21_sha256,
        "status": "NOT_IMPLEMENTED_WITHOUT_RAW_EXCEL_INPUT"
    }

with open(out_dir / "excel_unmatched_reclassification_v21.json", "w") as f:
    json.dump(unmatched_summary, f, indent=2)
print("Exported excel_unmatched_reclassification_v21.json.")

# 2. Causal Train-Only Eligibility Sensitivity Across Cutoffs
print("\n--- Causal Train-Only Eligibility Sensitivity Across Cutoffs ---")
cutoffs = {
    "F1": "2024-12-31",
    "F2": "2025-06-30",
    "F3": "2025-12-18",
    "F4": "2026-03-31"
}

# (a) Verified mode (fail-closed: missing publication dates withhold nonidentity merges)
sens_ver = evaluate_causal_train_eligibility_sensitivity(
    df_raw=df_panel_v21,
    alias_df=cand_alias_df,
    cutoffs=cutoffs,
    evidence_mode="verified"
)
print("\n(a) Verified Mode (Fail-Closed, 0 historical merges without gazette receipts):")
for f_name, f_data in sens_ver["cutoffs"].items():
    print(f"  {f_name} ({f_data['cutoff_date']}): Active Aliases={f_data['active_aliases_count']}, "
          f"Tier A Series={f_data['tier_a_series_count']}, Tier A Obs={f_data['tier_a_observations']:,}, "
          f"Subset Invariant Holds={f_data['subset_invariant_holds']}")

# (b) Assumed mode (sensitivity analysis on assumed observation handover dates)
sens_ass = evaluate_causal_train_eligibility_sensitivity(
    df_raw=df_panel_v21,
    alias_df=cand_alias_df,
    cutoffs=cutoffs,
    evidence_mode="assumed"
)
print("\n(b) Assumed Mode (Sensitivity to Nov 2025 APMC handover timing):")
for f_name, f_data in sens_ass["cutoffs"].items():
    print(f"  {f_name} ({f_data['cutoff_date']}): Active Aliases={f_data['active_aliases_count']}, "
          f"Tier A Series={f_data['tier_a_series_count']}, Tier A Obs={f_data['tier_a_observations']:,}, "
          f"Subset Invariant Holds={f_data['subset_invariant_holds']}")

# 3. Future Mutation Invariance Verification
print("\n--- Verifying Future Mutation Invariance ---")
sample_raw_for_inv = df_panel_v21.head(5000).copy()
inv_holds = verify_future_mutation_invariance(
    df_raw=sample_raw_for_inv,
    alias_df=cand_alias_df,
    cutoff_date="2025-12-18",
    evidence_mode="verified"
)
print(f"Future Mutation Invariance Verified: {inv_holds}")
assert inv_holds is True, "Future mutations corrupted historical cutoff evaluation!"


In [ ]:
# Cell 5: Phase 4: Export Forensic Artifacts & Manifest
print("\n=== PHASE 4: EXPORT FORENSIC ARTIFACTS & MANIFEST ===")

manifest_records = []
export_files = [
    out_dir / "identity_comparison_v2_vs_v21.csv",
    out_dir / "excel_unmatched_reclassification_v21.json",
    working_root / "mandi_5year_validated_v2.csv",
    working_root / "mandi_5year_validated_v21.csv"
]

for p in export_files:
    if p.exists():
        bytes_data = p.read_bytes()
        sha = hashlib.sha256(bytes_data).hexdigest()
        manifest_records.append({
            "filename": p.name,
            "path": str(p),
            "size_bytes": len(bytes_data),
            "sha256": sha
        })
        print(f"Exported {p.name}: {len(bytes_data):,} bytes, sha256={sha}")

manifest_v21_data = {
    "task_id": "MP-DATA-007",
    "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "legacy_v2_sha256": v2_sha256,
    "candidate_v21_sha256": v21_sha256,
    "files": manifest_records
}

with open(out_dir / "manifest_v21.json", "w") as f:
    json.dump(manifest_v21_data, f, indent=2)

print("\nSuccessfully generated manifest_v21.json.")
print("=== MP-DATA-007 COMPARISON EXECUTION COMPLETE ===")
